# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. All references to data entities (record sets, fields, columns) use their `@id` identifiers as defined in the Croissant schema.

### Dataset Source
The dataset source is provided via this Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata using `mlcroissant`. This will allow us to examine available record sets and fields for further analysis.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata fields (use attribute access, not indexing)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
List all available record sets (tables) and their fields using their `@id` references.

We use the metadata to navigate record sets and fields. This reveals the available data structures and prepares for targeted extraction.

In [ ]:
# Inspect available record sets and their fields
from pprint import pprint

def print_recordsets_and_fields(ds):
    print("Available Record Sets and Fields (by @id):\n")
    for recordset in ds.metadata.record_sets:
        print(f"RecordSet name: {recordset.name}")
        print(f"  @id: {recordset.id}")
        print("  Fields:")
        for field in recordset.fields:
            field_id = getattr(field, "id", "<no_id>")
            print(f"    - {field.name} (id: {field_id}, type: {field.data_type})")
        print()

print_recordsets_and_fields(dataset)

## 3. Data Extraction
We demonstrate how to load records from the main record set (data table). 
Replace the variable values as needed based on the above-overviewed `@id` list.

All access and extraction is by `@id`.

In [ ]:
# List all record_set @ids for quick reference
record_set_ids = [r.id for r in dataset.metadata.record_sets]
print(f"Available record set @ids: {record_set_ids}")

# For this dataset, select the main table/record set.
# The actual @id varies; using the first as example.
main_record_set_id = record_set_ids[0]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Shape: {dataframes[record_set_id].shape}\n---")

# Show fields (columns) in the main record set
print(f"Columns for '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply some processing steps – filtering by a numeric field, normalizing its values, and grouping by a categorical field.

All field selections use their canonical `@id`.

In [ ]:
# Example: choose a numeric field and a categorical (group-by) field
# Uncomment and adapt the fields as needed based on the metadata inspection

# Fetch the record set object
main_recordset = [r for r in dataset.metadata.record_sets if r.id == main_record_set_id][0]
# List all field ids for manual inspection
field_ids = [f.id for f in main_recordset.fields]
print("Field ids in main record set:")
pprint(field_ids)

# Let's pick a numeric field (change as seen appropriate!):
# Example: Assume there's a field called 'cr:Abundance' and a categorical field 'cr:Description'
# Update these according to real field ids listed above.
numeric_field_id = field_ids[2]  # for example purposes--should be changed after inspecting above
group_field_id = field_ids[1]    # for example purposes--should be changed

df = dataframes[main_record_set_id]
# Convert numeric field to float (if needed)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter
threshold = df[numeric_field_id].mean()  # example: above-mean
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Records with numerical field '{numeric_field_id}' > mean ({threshold:.2f}):")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by
if group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nMean of '{numeric_field_id}' by group '{group_field_id}':")
    print(grouped.head())

## 5. Visualization
We now plot the distribution of the chosen numeric field, and the grouped means by the categorical field. All axes and legends use schema `@id`s for traceability.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Histogram of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouped means are available, plot them
if group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
    plt.figure(figsize=(10, 5))
    grouped.head(20).plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} by {group_field_id} (top 20)")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load metadata and records from a FAIR² dataset using `mlcroissant`, inspect record sets and fields (by `@id`), extract records for analysis, apply basic EDA (filtering, normalization, grouping), and plot results.

All entity references follow the Croissant schema's `@id` system, supporting reproducible FAIR data workflows. For more advanced analyses, see [mlcroissant documentation](https://mlcommons.org/croissant/).
